# Task 1.1 — Core Contribution / Architecture (8 marks)

**Paper**: *Gaussian Processes for Time-Marked Time-Series Data*  
**Authors**: John P. Cunningham, Zoubin Ghahramani, Carl E. Rasmussen  
**Venue**: AISTATS 2012  
**Roll Number**: 230035 — Karthik Reddy

## Step-by-Step Method Description

### Step 1: Define the Time-Marked Input Space

- **Description**: Given a collection of N time series $\{y^{(n)}(t)\}$ where each series has K known event markers $\{m_1^{(n)}, m_2^{(n)}, \ldots, m_K^{(n)}\}$, the method re-represents each observation time $t_i$ as a (K+1)-dimensional vector. Specifically, for an observation at absolute time $t_i$ in series $n$, the input becomes the vector of *relative times*: $(t_i - m_1^{(n)}, t_i - m_2^{(n)}, \ldots, t_i - m_K^{(n)})$. This transforms a 1-dimensional time series regression problem into a multi-dimensional GP regression problem.
- **Reference**: Equation 1 (Section 2) — the time-marked covariance $k_{TM}$ is defined over these relative-time vectors.
- **Purpose**: By re-expressing time relative to each event marker, the model can learn how the signal depends on the *timing of events*, not just absolute time. This is the foundational design choice that enables the GP to capture event-driven dynamics that standard GP regression would miss.

### Step 2: Construct the Time-Marked Covariance Kernel

- **Description**: A standard stationary kernel (e.g., squared exponential) is applied in the expanded input space. Concretely, the kernel uses per-dimension lengthscales $l_k$, so:

$$k_{TM}(t_i^{(p)}, t_j^{(q)}) = \sigma^2 \exp\left(-\sum_{k=1}^{K} \frac{1}{2l_k^2} \left[(t_i^{(p)} - m_k^{(p)}) - (t_j^{(q)} - m_k^{(q)})\right]^2\right)$$

  Each lengthscale $l_k$ controls how strongly the signal depends on marker $k$. A small $l_k$ means the signal changes rapidly around marker $k$; a large $l_k$ means marker $k$ has little influence.
- **Reference**: Equation 2 (Section 2).
- **Purpose**: This is the paper's key technical contribution — it defines a valid positive-definite kernel that naturally incorporates event timing. Because it's a standard kernel on a transformed input, all GP machinery (posterior inference, marginal likelihood, etc.) works out of the box.

### Step 3: Enforce Causality via Input Warping

- **Description**: In many real scenarios, the signal should only respond *after* an event occurs, not before. The paper enforces this by warping each input dimension with the function $h(t) = t \cdot \mathbb{I}(t > 0)$ (the positive part). Applied to the time-marked inputs, this means: if an observation occurs *before* marker $k$ (i.e., $t_i - m_k < 0$), the warped input is 0 for that dimension. The effect is that all pre-marker time points map to the same input value, so the GP predicts the same (flat) output for all of them — the signal cannot "anticipate" the event.
- **Reference**: Section 2.1 and Equation 3 — the causal GP is defined via this warping, and the paper proves it is equivalent to a specific linear filter $g(t, u)$.
- **Purpose**: This addresses a fundamental issue: standard GP regression is symmetric in time (acausal), but many physical processes are causal. By warping inputs, the model structurally prevents future events from influencing past observations, which improves generalization on causal data.

### Step 4: GP Posterior Inference

- **Description**: With the causal time-marked covariance kernel defined, standard GP regression is applied. Given training data (N-1 time series in LOOCV), the GP posterior mean and variance are computed for test points:

$$\mu_* = K_{*,\text{train}} (K_{\text{train,train}} + \sigma_n^2 I)^{-1} y_{\text{train}}$$
$$\sigma_*^2 = K_{*,*} - K_{*,\text{train}} (K_{\text{train,train}} + \sigma_n^2 I)^{-1} K_{\text{train},*}$$

  where all $K$ matrices use the causal time-marked kernel. The paper also adds temporally colored noise (modeled as an additional GP noise process) rather than just white noise.
- **Reference**: Section 2 (final paragraphs) and Section 3 — defers to Rasmussen & Williams (2006) for standard GP inference.
- **Purpose**: Because the time-marked and causal GP model is simply a particular choice of covariance, the model inherits all conventional GP machinery — no special inference algorithm is needed.

### Step 5: Hyperparameter Learning via Marginal Likelihood

- **Description**: The kernel hyperparameters (signal variance $\sigma^2$, lengthscales $l_1, \ldots, l_K$, and noise parameters) are learned by maximizing the log marginal likelihood of the training data. This is done via gradient-based optimization (L-BFGS or similar). The lengthscales $l_k$ learned for each marker dimension serve double duty: they both improve prediction *and* reveal which markers are most relevant to the signal (Automatic Relevance Determination).
- **Reference**: Section 3 (Experimental Framework) — hyperparameters are optimized on the LOOCV training set, with results averaged over 10 random initializations.
- **Purpose**: This step adapts the model to the data. The ARD property of the learned lengthscales is particularly valuable: if the learned $l_k$ for marker $k$ is very large, it means that marker contributes little to the signal — effectively providing interpretable feature importance.

### Step 6: Evaluation via Leave-One-Out Cross-Validation (LOOCV)

- **Description**: Performance is evaluated by holding out one complete time series at a time, training the GP on the remaining N-1 series, and predicting the held-out series. The prediction quality is measured by RMSE between the GP posterior mean and the true observations, averaged across all N held-out series.
- **Reference**: Section 3 and Table 1 — RMSE is used for all comparisons.
- **Purpose**: LOOCV provides a fair comparison across all model variants (standard GP, clipping, acausal TM-GP, causal TM-GP) and non-probabilistic baselines (averaging). It tests generalization to unseen trials.

## Final Summary Sentence

This paper solves the problem of regression on time-series data influenced by known temporal events (time markers) by re-representing observations in a multi-dimensional event-relative input space and introducing a causality-enforcing input warping, and the authors claim this approach is superior to alternatives because it naturally and principled-ly incorporates event timing into the GP covariance function — without heuristic data clipping or any changes to standard GP inference — while the learned per-marker lengthscales provide automatic relevance determination of each event's influence on the signal.